# Gradient boosting

[Random forests](random-forests.ipynb) build many trees **independently, in
parallel**, and average them (bagging). **Boosting** is the opposite mechanism:
it builds trees **sequentially**, each one trained to correct the errors the
previous trees left behind. Stated plainly before any code — that difference is
the whole idea.

```{warning}
**Ecosystem reality.** Gradient boosting in Rust is essentially a *single-crate*
story — [`perpetual`](https://github.com/perpetual-ml/perpetual) — with nothing
like the breadth of Python's XGBoost / LightGBM / CatBoost. And in this image
`perpetual` doesn't build: it **requires nightly Rust** (`#![feature(...)]`),
while we pin a stable toolchain for reproducibility. So rather than ship a
broken dependency, we implement the boosting mechanism **by hand** on
`smartcore` regression trees — which is also the clearest way to see how it
works.
```

## Boosting, by hand

Start from a constant prediction (the mean). Each round: compute the
**residuals** (what we got wrong), fit a shallow "weak" tree to *those*, and add
a small fraction (the learning rate) of its predictions. The training error
should fall each round:

In [ ]:
:dep smartcore = { version = "0.3" }
{
    use smartcore::api::SupervisedEstimator;
    use smartcore::linalg::basic::matrix::DenseMatrix;
    use smartcore::tree::decision_tree_regressor::{DecisionTreeRegressor, DecisionTreeRegressorParameters};

    // A nonlinear target a single shallow tree can't capture well alone.
    let rx: Vec<Vec<f64>> = (0..120).map(|i| vec![i as f64 * 0.1]).collect();
    let ry: Vec<f64> = rx.iter().map(|r| (r[0]).sin() * 3.0 + r[0] * 0.4).collect();
    let x = DenseMatrix::new(rx.len(), 1, rx.iter().flatten().cloned().collect(), false);

    let rmse = |f: &[f64]| (ry.iter().zip(f).map(|(a, p)| (a - p).powi(2)).sum::<f64>() / ry.len() as f64).sqrt();
    let learning_rate = 0.3;
    let mut f = vec![ry.iter().sum::<f64>() / ry.len() as f64; ry.len()];   // F0 = mean
    println!("round 0 (constant mean): RMSE = {:.3}", rmse(&f));

    for round in 1..=8 {
        let residuals: Vec<f64> = ry.iter().zip(&f).map(|(a, p)| a - p).collect();
        let weak = DecisionTreeRegressor::fit(
            &x, &residuals,
            DecisionTreeRegressorParameters::default().with_max_depth(3),
        ).unwrap();
        let update = weak.predict(&x).unwrap();
        for i in 0..f.len() { f[i] += learning_rate * update[i]; }
        println!("round {}: RMSE = {:.3}", round, rmse(&f));
    }
}

Each round chips away at the error a single tree couldn't — that's boosting.

## Boosting vs. bagging

| | Random forest (bagging) | Gradient boosting |
| --- | --- | --- |
| Trees built | independently, **in parallel** | sequentially, **each fixes the last** |
| Reduces mainly | variance | bias |
| Parallelism | trivial (free in `smartcore`) | limited (sequential by nature) |
| Overfitting risk | low | higher — needs the learning rate + few rounds as control |

A production library adds regularization, second-order gradients, and clever
tree construction on top of this loop. In Rust today that library is
`perpetual` — powerful (it advertises built-in feature importance and ONNX
export) but, as noted above, **nightly-only right now**, so re-check its
toolchain status before depending on it.

Next: [bagging & stacking](bagging-and-stacking.ipynb) — combining *different*
model types by hand, and asking whether an ensemble beats its best member.